# 🛡️ EXP-011: Occlusion-Resistant Multi-Joint Kinematic Heuristics

Notebook ini mengeksplorasi **Ketahanan terhadap Oklusi Furnitur (Occlusion Robustness)**:
Di kamar tidur lansia nyata, tubuh sering terhalang sebagian oleh:
- Ranjang tempat tidur (menutup kaki/lutut)
- Meja samping / nakas (menutup pinggul)
- Selimut / bantal

### 🎯 Tujuan Eksperimen:
Membandingkan **Baseline Single-Pair Pose (EXP-009)** yang hanya mengandalkan sendi bahu-ke-pinggul vs **Hierarchical Multi-Joint Fall Estimator (EXP-011)** yang memiliki pohon keputusan fallback adaptif (Tier-1: Bahu-Pinggul, Tier-2: Kepala-Bahu, Tier-3: Pinggul-Kaki, Tier-4: Axis Kepala-Kaki, Tier-5: Aspect Ratio) di bawah berbagai skenario oklusi.

In [ ]:
import math
import time
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("🚀 Memuat model YOLO11-Pose...")
model = YOLO("yolo11n-pose.pt")
print("✅ Model siap!")

In [ ]:
def vector_angle_to_horizontal(p1, p2) -> float:
    dx = p1[0] - p2[0]
    dy = p1[1] - p2[1]
    return math.degrees(math.atan2(abs(dy), abs(dx)))

def baseline_spine_angle(kpts: np.ndarray, ar: float):
    sh_conf = (kpts[5][2] + kpts[6][2]) / 2.0
    hip_conf = (kpts[11][2] + kpts[12][2]) / 2.0
    if sh_conf > 0.25 and hip_conf > 0.25:
        mid_sh = ((kpts[5][0] + kpts[6][0]) / 2.0, (kpts[5][1] + kpts[6][1]) / 2.0)
        mid_hip = ((kpts[11][0] + kpts[12][0]) / 2.0, (kpts[11][1] + kpts[12][1]) / 2.0)
        return vector_angle_to_horizontal(mid_sh, mid_hip), "Shoulder-Hip"
    return (90.0 if ar < 0.8 else (20.0 if ar > 1.15 else 50.0)), "BBox-Fallback"

def hierarchical_angle_estimator(kpts: np.ndarray, ar: float):
    head_conf = (kpts[0][2] + kpts[1][2] + kpts[2][2]) / 3.0
    sh_conf = (kpts[5][2] + kpts[6][2]) / 2.0
    hip_conf = (kpts[11][2] + kpts[12][2]) / 2.0
    knee_conf = (kpts[13][2] + kpts[14][2]) / 2.0
    ankle_conf = (kpts[15][2] + kpts[16][2]) / 2.0

    if sh_conf > 0.25 and hip_conf > 0.25:
        mid_sh = ((kpts[5][0] + kpts[6][0]) / 2.0, (kpts[5][1] + kpts[6][1]) / 2.0)
        mid_hip = ((kpts[11][0] + kpts[12][0]) / 2.0, (kpts[11][1] + kpts[12][1]) / 2.0)
        return vector_angle_to_horizontal(mid_sh, mid_hip), "Tier-1 (Spine)"

    if head_conf > 0.20 and sh_conf > 0.25:
        head_pt = (kpts[0][0], kpts[0][1])
        mid_sh = ((kpts[5][0] + kpts[6][0]) / 2.0, (kpts[5][1] + kpts[6][1]) / 2.0)
        return vector_angle_to_horizontal(head_pt, mid_sh), "Tier-2 (Head-Shoulder)"

    if hip_conf > 0.25 and (knee_conf > 0.20 or ankle_conf > 0.20):
        mid_hip = ((kpts[11][0] + kpts[12][0]) / 2.0, (kpts[11][1] + kpts[12][1]) / 2.0)
        p_lower = ((kpts[13][0] + kpts[14][0]) / 2.0, (kpts[13][1] + kpts[14][1]) / 2.0) if knee_conf > 0.20 else \
                  ((kpts[15][0] + kpts[16][0]) / 2.0, (kpts[15][1] + kpts[16][1]) / 2.0)
        return vector_angle_to_horizontal(mid_hip, p_lower), "Tier-3 (Hip-Legs)"

    if head_conf > 0.20 and ankle_conf > 0.20:
        head_pt = (kpts[0][0], kpts[0][1])
        mid_ank = ((kpts[15][0] + kpts[16][0]) / 2.0, (kpts[15][1] + kpts[16][1]) / 2.0)
        return vector_angle_to_horizontal(head_pt, mid_ank), "Tier-4 (Head-Ankle Axis)"

    return (90.0 if ar < 0.8 else (15.0 if ar > 1.15 else 50.0)), "Tier-5 (Aspect Ratio)"

In [ ]:
dataset_dir = Path("../data/processed") if Path("../data/processed").exists() else Path("ml/data/processed")
test_img_dir = dataset_dir / "images" / "test"

def evaluate(occlusion_mode="none"):
    img_files = sorted(list(test_img_dir.glob("*.jpg")) + list(test_img_dir.glob("*.png")))
    b_corr, h_corr, total = 0, 0, 0
    for img_p in img_files:
        is_fall = 1 if img_p.name.startswith("fall") else 0
        res = model(str(img_p), verbose=False, conf=0.25)
        if not res or len(res[0].boxes) == 0: continue
        boxes = res[0].boxes
        best_idx = int(boxes.conf.argmax().item())
        kpts = res[0].keypoints.data[best_idx].cpu().numpy().copy()
        bbox = boxes.xyxy[best_idx].tolist()
        ar = max(1.0, bbox[2] - bbox[0]) / max(1.0, bbox[3] - bbox[1])
        
        if occlusion_mode == "lower":
            for idx in [11, 12, 13, 14, 15, 16]: kpts[idx][2] = 0.0
        elif occlusion_mode == "hips":
            for idx in [11, 12]: kpts[idx][2] = 0.0
            
        b_ang, _ = baseline_spine_angle(kpts, ar)
        h_ang, _ = hierarchical_angle_estimator(kpts, ar)
        if (1 if b_ang < 30.0 else 0) == is_fall: b_corr += 1
        if (1 if h_ang < 30.0 else 0) == is_fall: h_corr += 1
        total += 1
    return b_corr / total, h_corr / total, total

scenarios = [("Full View", "none"), ("Hip Occlusion", "hips"), ("Lower-Body Occluded", "lower")]
results = []
for name, mode in scenarios:
    b_a, h_a, tot = evaluate(mode)
    results.append((name, b_a*100, h_a*100, (h_a - b_a)*100))
    print(f"{name:<22}: Baseline={b_a*100:.2f}%, Hierarchical={h_a*100:.2f}%, Gain={(h_a - b_a)*100:+.2f}%")

In [ ]:
labels = [r[0] for r in results]
b_scores = [r[1] for r in results]
h_scores = [r[2] for r in results]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
rects1 = ax.bar(x - width/2, b_scores, width, label='Baseline Pose (EXP-009)', color='#e74c3c')
rects2 = ax.bar(x + width/2, h_scores, width, label='Hierarchical Multi-Joint (EXP-011)', color='#2ecc71')

ax.set_ylabel('Akurasi Klasifikasi Postur (%)', fontsize=11)
ax.set_title('Ketahanan Model terhadap Oklusi Objek Furnitur (EXP-011)', fontsize=12, weight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 100)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()